In [1]:
from pathlib import Path
import sys

import numpy as np
from scipy import stats

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    for parent in ROOT.parents:
        if (parent / "src").exists() and (parent / "data").exists():
            ROOT = parent
            break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.analysis import add_futures, clean_data, sorting
from src.data_loader import load_data

In [2]:
path = ROOT / "data" / "db.csv"
df = load_data(path, show_info=False)
df = clean_data(df)
df = add_futures(df)
df = sorting(df)
df.head()

,order_id,user_id,date,region,city,platform,traffic_source,category,product_name,price,quantity,discount,payment_method,delivery_days,is_returned,rating,revenue,revenue_discnt
320,50321,1006,2024-04-27,Voronezh Oblast,Voronezh,App,Social,electronics,iPhone 14,931.97,1,0,Card,4,0,4.4,931.97,931.9700
109,50110,1168,2024-02-10,Voronezh Oblast,Voronezh,Web,Direct,electronics,Samsung Galaxy S23,846.31,1,3,SBP,3,0,4.9,846.31,820.9207
178,50179,1051,2024-03-06,Voronezh Oblast,Voronezh,Web,Paid Search,electronics,Samsung Galaxy S23,808.80,1,3,Card,3,0,4.2,808.80,784.5360
23,50024,1106,2024-01-09,Voronezh Oblast,Voronezh,App,Direct,electronics,Laptop Lenovo IdeaPad,736.75,1,10,Installments,2,0,4.9,736.75,663.0750
295,50296,1214,2024-04-17,Voronezh Oblast,Voronezh,App,Social,clothes,Winter Coat,193.60,2,15,Card,3,0,4.6,387.20,329.1200


In [3]:
arr = np.array(df["price"])
print("count: ", arr.size)
print("mean: ", np.mean(arr))
print("median: ", np.median(arr))
print("var: ", np.var(arr))
print("q50: ", np.quantile(arr, 0.5))
print("quartiles: ", np.quantile(arr, [0.25, 0.5, 0.75]))

count:  380
mean:  244.61397368421052
median:  101.88499999999999
var:  83484.63824184141
q50:  101.88499999999999
quartiles:  [ 55.02   101.885  249.5725]


In [4]:
print(df[df["platform"] == "App"])
print(df[df["platform"] == "Web"])

     order_id  user_id        date           region      city platform  \
320     50321     1006  2024-04-27  Voronezh Oblast  Voronezh      App   
23      50024     1106  2024-01-09  Voronezh Oblast  Voronezh      App   
295     50296     1214  2024-04-17  Voronezh Oblast  Voronezh      App   
161     50162     1122  2024-02-29  Voronezh Oblast  Voronezh      App   
181     50182     1166  2024-03-07  Voronezh Oblast  Voronezh      App   
..        ...      ...         ...              ...       ...      ...   
64      50065     1038  2024-01-26    Bashkortostan       Ufa      App   
307     50308     1074  2024-04-22    Bashkortostan       Ufa      App   
12      50013     1105  2024-01-06    Bashkortostan       Ufa      App   
350     50351     1188  2024-05-08    Bashkortostan       Ufa      App   
299     50300     1150  2024-04-19    Bashkortostan       Ufa      App   

     traffic_source     category           product_name   price  quantity  \
320          Social  electronics  

In [5]:
revenue_arr = np.array(df["revenue_discnt"])
print(revenue_arr[np.argmax(revenue_arr)])
print(revenue_arr[revenue_arr > 200])

1935.063
[ 931.97    820.9207  784.536   663.075   329.12    250.06    897.08
  863.512   216.8752  206.8     200.0768  808.6722  808.0163  799.19
  784.1764  782.7066  747.137   734.3882  587.43    556.1856  283.9746
  248.3879  232.594   796.5312  765.225   756.8813  711.8776  670.0255
  527.8656  487.1867  382.4672  293.3262  261.8847  205.3568  204.748
  201.936  1603.5648  811.7934  704.5056  695.2128  502.2017  481.1672
  384.264   336.168   315.9975  295.596   252.4     250.5412  227.6865
  212.04   1098.237   844.58    765.7252  748.72    717.1449  711.8442
  702.693   690.777   634.92    247.9344  244.692   233.5872  213.6714
  211.99   1852.1196  813.3615  747.312   731.2512  730.6002  659.0094
  572.6248  520.9515  357.7752  204.8834  741.0676  685.1868  529.776
  433.4688  328.788   249.41    230.2944  228.366   205.1082  802.0831
  795.948   467.6337  240.6888  226.0794  907.4641  895.3017  853.7649
  813.599   674.0345  636.17    499.605   478.6338  312.13    261.681
  20

In [6]:
generator_arr = np.fromiter((value for value in df["revenue_discnt"]), dtype=float)
print("generator mean revenue: ", np.mean(generator_arr))
print("generator median revenue: ", np.median(generator_arr))

generator mean revenue:  278.5046147368421
generator median revenue:  158.74275


In [7]:
t_stat, p_value = stats.ttest_1samp(revenue_arr, 150)
print(t_stat, p_value)

app = np.array(df[df["platform"] == "App"]["revenue_discnt"].dropna())
web = np.array(df[df["platform"] == "Web"]["revenue_discnt"].dropna())
stat, p_value = stats.mannwhitneyu(app, web)
print(stat, p_value)

8.236249420419789 2.9148600871559397e-15
7181.0 0.0004999434876825036
